In [1]:
import operator
import string
import jieba
import pandas as pd
import re
from ast import literal_eval
import jieba.posseg as psg
import jieba.posseg as pseg   # 分词处理
from pyltp import SentenceSplitter
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer # 计算tfidf值
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', 1000)#加了这一行那表格的一行就不会分段出现了
#显示所有列
pd.set_option('display.max_columns', None)
#显示所有行
pd.set_option('display.max_rows', None)


In [2]:
# 主题分析

#定义提取各个平台中评论对象主题的函数
def print_top_words(model, feature_names, n_top_words):
    tword = []
    i=1
    # 遍历每个主题，获取每个主题的关键词
    for topic_idx, topic in enumerate(model.components_):
        # 获取每个主题的前 n_top_words 个关键词，并将它们以空格连接成一个字符串
        topic_w = " ".join([feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]])
        # 将主题编号和关键词保存到字典中，并将字典添加到列表tword中
        tempDict = {}
        tempDict["topicId"] = f"主题 {i}"
        tempDict["topWord"] = topic_w
        tword.append(tempDict)
        i += 1
    # 返回一个列表，其中包含了每个主题的编号和关键词
    return tword


In [3]:

# n_features: 提取的特征词语个数
# n_topics：主题数量
# max_iter：算法最大迭代次数
# n_top_words：每个主题根据关键词的出现概率显示前n个关键词

def LDAAnalysis(n_features, n_topics, max_iter, n_top_words, data):
    n_features = n_features  # 提取500个特征词语

    # 构建一个CountVectorizer对象，用于将文本数据转换为词频矩阵
    tf_vectorizer = CountVectorizer(max_features=n_features,  #最多提取的特征词语数量
                                    stop_words="english", # 停用词
                                    max_df=0.8, # 剔除过于频繁出现的词语
                                    min_df=2,   # 剔除过于少出现的词语
                                    encoding="utf-8",
                                    )
     # 将文本数据转换为词频矩阵
    tf = tf_vectorizer.fit_transform(data)

    n_topics = n_topics  # 主题数量

     # 构建对象，用于执行LDA主题模型
    lda = LatentDirichletAllocation(n_components=n_topics,
                                    max_iter=max_iter,  # 最大迭代数量
                                    learning_method="batch",  # 学习方法，小批量
                                    learning_offset=10,
                                    doc_topic_prior=0.1,
                                    topic_word_prior=0.01,
                                    random_state=0
                                    )
   
    # 在词频矩阵上执行LDA主题模型
    lda.fit(tf)
    n_top_words = n_top_words
    tf_feature_names = tf_vectorizer.get_feature_names()
     # 每个主题中，根据关键词出现概率显示前n_top_words个关键词
    topic_word = print_top_words(lda, tf_feature_names, n_top_words)
    # 返回每个主题中前n_top_words个关键词
    return topic_word


In [4]:
# 提取各个平台对ChatGPT和文心一言这两个对象的评论
bjh_DF = pd.read_csv("../Data/merged_file_bjh.csv", encoding="utf-8")
bz_wxyy_DF = pd.read_csv("../Data/merged_file_bz_wxyy.csv", encoding="utf-8")
bz_gpt_DF = pd.read_csv("../Data/merged_file_bz_gpt.csv", encoding="utf-8")
wb_wxyy_DF = pd.read_csv("../Data/merged_file_wb_wxyy.csv", encoding="utf-8")
wb_gpt_DF = pd.read_csv("../Data/merged_file_wb_gpt.csv", encoding="utf-8")
zh_wxyy_DF = pd.read_csv("../Data/merged_file_zh_wxyy.csv", encoding="utf-8")
zh_gpt_DF = pd.read_csv("../Data/merged_file_zh_gpt.csv", encoding="utf-8")


In [5]:
#定义一个函数theme打印主题关键词
def theme(DF_name):
        
    pos_data = pd.DataFrame()
    neg_data = pd.DataFrame()

    # 将正向评论的文本分词结果存储在pos_data中
    pos_data["文本分词"] = DF_name["文本分词"][DF_name['评论情感得分'] > 0]
    # 将负向评论的文本分词结果存储在neg_data中
    neg_data["文本分词"] = DF_name["文本分词"][DF_name['评论情感得分'] < 0]
    # 对正向评论进行LDA主题模型分析
    pos_result = LDAAnalysis(500, 3, 100, 10, pos_data["文本分词"])
    # 对负向评论进行LDA主题模型分析
    neg_result = LDAAnalysis(500, 3, 100, 10, neg_data["文本分词"])
    #打印关键词
    print(pos_result)
    print(neg_result)


In [9]:
theme(bjh_DF)
theme(zh_gpt_DF)
theme(zh_wxyy_DF)
theme(bz_gpt_DF)
theme(bz_wxyy_DF)
theme(wb_gpt_DF)
theme(wb_wxyy_DF)

c:\Users\Universal\anaconda3\lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


[{'topicId': '主题 1', 'topWord': '用户 技术 创作 语言 内容 领域 智能 文本 对话 人工智能'}, {'topicId': '主题 2', 'topWord': '产品 企业 智能 技术 服务 行业 平台 数据 能力 数字'}, {'topicId': '主题 3', 'topWord': '人工智能 通义 技术 公司 发展 领域 文化 人类 产业 能力'}]
[{'topicId': '主题 1', 'topWord': '人工智能 生活 工作 消息 微信 未来 名字 时间 故事 意识'}, {'topicId': '主题 2', 'topWord': '创作 故事 关键词 时间 意识 名字 未来 工作 生活 消息'}, {'topicId': '主题 3', 'topWord': '生活 消息 未来 时间 故事 意识 微信 工作 名字 创作'}]


c:\Users\Universal\anaconda3\lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


[{'topicId': '主题 1', 'topWord': '工作 人类 内容 工具 现在 技术 人工智能 信息 发展 未来'}, {'topicId': '主题 2', 'topWord': '语言 数据 文本 对话 用户 智能 训练 技术 任务 能力'}, {'topicId': '主题 3', 'topWord': '人类 训练 能力 例子 过程 人工智能 作文 习惯 社会 数据'}]
[{'topicId': '主题 1', 'topWord': '公司 人类 用户 数据 能力 市场 故事 事情 中文 免费'}, {'topicId': '主题 2', 'topWord': '语言 人类 现在 技术 发展 人工智能 文字 能力 玩意 创作'}, {'topicId': '主题 3', 'topWord': '工业革命 现在 未来 插件 代码 互联网 人工 想象 英语 时间'}]


c:\Users\Universal\anaconda3\lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


...

[{'topicId': '主题 1', 'topWord': '数据 训练 中文 公司 广告 现在 产品 互联网 计算器 英文'}, {'topicId': '主题 2', 'topWord': '技术 现在 发展 社会 人类 岗位 科技 生产力 人工智能 生产'}, {'topicId': '主题 3', 'topWord': '人类 工具 工作 能力 用户 语言 未来 体验 视频 感觉'}]
[{'topicId': '主题 1', 'topWord': '数据 现在 产品 中文 开源 训练 互联网 公司 能力 工具'}, {'topicId': '主题 2', 'topWord': '人类 计算器 人工智能 现在 视频 研究 论文 医院 老板 错误'}, {'topicId': '主题 3', 'topWord': '技术 现在 广告 公司 发展 工作 生产力 科技 时间 目前'}]


c:\Users\Universal\anaconda3\lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


[{'topicId': '主题 1', 'topWord': '公司 技术 人工智能 产品 科技 企业 发展 算力 领域 服务'}, {'topicId': '主题 2', 'topWord': '板块 科技 股份 市场 概念 指数 个股 学生 资金 创业板'}, {'topicId': '主题 3', 'topWord': '人工智能 人类 技术 能力 用户 工作 语言 机器人 内容 聊天'}]
[{'topicId': '主题 1', 'topWord': '个人 测试 发布会 港股 办公 邮件 基本上 人用 发邮件 外界'}, {'topicId': '主题 2', 'topWord': '语言 方向 大厂 人工智能 企业家 科技 人类 公司 现在 意义'}, {'topicId': '主题 3', 'topWord': '人工智能 用户 人类 技术 能力 聊天 现在 中文 公司 工作'}]


c:\Users\Universal\anaconda3\lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


[{'topicId': '主题 1', 'topWord': '公司 技术 科技 市场 发展 股份 数字 场景 产品 国际'}, {'topicId': '主题 2', 'topWord': '智能 产品 发布会 人工智能 企业 测试 服务 用户 技术 语言'}, {'topicId': '主题 3', 'topWord': '中文 能力 发布会 语言 产品 创作 新闻 首席 创始人 用户'}]
[{'topicId': '主题 1', 'topWord': '发布会 中文 现在 产品 技术 能力 公司 企业 市场 人工智能'}, {'topicId': '主题 2', 'topWord': '孩子 企业家 港股 产品 测试 资格 发布会 今天 企业 视频'}, {'topicId': '主题 3', 'topWord': '外界 人类 老婆 有点 预期 近日 软件 人工 科技 人工智能'}]
